### Core Function Collection playground
In this script all the currently up-to-date essential functions for the algorithm are listed & roughly explained 

In [57]:
# Minimal Function requirements: 
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from typing import Tuple

# Plotting 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from itertools import combinations

### Functions for the Algorithm: 
- Pairwise squared Distances 
- KNN Graph for Transition matrix calculation (Sigma Hyperparamter)
- KNN Graph for Transition matrices Calculation (Sigma tuned by Perplexity)
- KL-Divergence Calculation 
- MMD^2 Calculation 
- Cauchy Kernel (Gamma Hyperparamter)

I put them all here into a single cell, you might consider putting them into an .py file and import them just like that so it doesnt look so nooby

In [ ]:
def pairwise_squared_distances(x: torch.Tensor) -> torch.Tensor:
    """
    Return the NxN matrix of squared Euclidean distances.
    x: (N, D) tensor
    """
    # torch.cdist returns Euclidean distance; square it for squared distances.
    return torch.cdist(x, x, p=2.0) ** 2

def knn_graph(x, k=10, sigma=None):
    # Returns adjacency (n,n) with exponential decay weights on kNN
    with torch.no_grad():
        dist = pairwise_distances(x)
        knn_idx = dist.topk(k+1, largest=False)[1][:,1:] 
        n = x.size(0)
        W = torch.zeros(n, n, device=x.device)
        if sigma is None:
            sigma = dist.median().item()**0.5 + 1e-8  # bandwidth heuristic
        for i in range(n):
            neighbors = knn_idx[i]
            for j in neighbors:
                W[i,j] = torch.exp(-dist[i,j]/(2*sigma**2))
        # Row normalize to probabilities: 
        W = W / W.sum(dim=1, keepdim=True)
    return W

def knn_graph_with_perplexity(
    x: torch.Tensor,
    perplexity: float = 30.0,
    k: int = None,
    tol: float = 1e-5,
    max_iter: int = 50,
    sym_mode: str = "mean",   # "mean", "sum", or "none"
    use_probabilities: bool = True,  # if False: use unnormalized kernel weights exp(-beta * d2)
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Build a KNN graph whose edge weights are Gaussian-kernelized with a
    node-specific bandwidth sigma_i chosen by matching the local perplexity.

    Args
    ----
    x : (N, D) float tensor on any device.
    perplexity : desired perplexity per node (effective number of neighbors).
    k : number of neighbors to keep per node. If None, set k = min(N-1, max(10, 3*perplexity)).
    tol : tolerance on entropy matching.
    max_iter : max binary search iterations for beta (= 1/(2*sigma^2)).
    sym_mode : how to symmetrize the directed P(i|j):
        - "mean": (W + W^T)/2  (default; common for similarity graphs)
        - "sum" :  W + W^T
        - "none":  keep directed graph (sparse, not symmetric)
    use_probabilities : if True, store row-stochastic P(j|i) on KNN; else store raw kernel e^{-beta*d^2}.

    Returns
    -------
    W : sparse COO tensor (N, N) with KNN weights (symmetrized if requested).
    sigmas : (N,) tensor of local bandwidths sigma_i.
    knn_indices : (N, k) long tensor with neighbor indices per node.
    """
    assert x.dim() == 2, "x must be (N, D)"
    N = x.size(0)
    device = x.device
    dtype = x.dtype

    # Choose k if not provided; use 3*perplexity as a practical default (like t-SNE heuristics).
    if k is None:
        k = int(max(10, 3 * perplexity))
    k = min(k, N - 1)  # cannot exceed N-1

    # Precompute pairwise squared distances
    d2 = pairwise_squared_distances(x)  # (N, N)

    # Mask self-distance so it is never picked as a neighbor
    d2_no_diag = d2.clone()
    d2_no_diag.fill_diagonal_(float("inf"))

    # Get K nearest neighbors (distances + indices)
    nn_dists, nn_indices = torch.topk(d2_no_diag, k=k, largest=False)  # both (N, k)

    log_perp = torch.log(torch.tensor(perplexity, device=device, dtype=dtype))

    # Storage
    sigmas = torch.zeros(N, device=device, dtype=dtype)

    # To build a sparse matrix: collect COO triplets
    rows = []
    cols = []
    vals = []

    # Binary search per node to match entropy to log(perplexity)
    for i in range(N):
        Di = nn_dists[i]     # (k,) squared distances to the k nearest neighbors of i
        idx = nn_indices[i]  # (k,) neighbor indices

        beta = torch.tensor(1.0, device=device, dtype=dtype)  # beta = 1/(2*sigma^2)
        betamin = None
        betamax = None

        # Handle degenerate case where all distances are identical (very rare)
        if torch.allclose(Di, Di[0]):
            # Any beta works; keep beta=1.0 and continue
            pass
        else:
            for _ in range(max_iter):
                # Compute row-conditional probabilities (restricted to KNN)
                Pi = torch.exp(-Di * beta)  # unnormalized
                Pi_sum = Pi.sum()
                # Guard against numerical underflow
                if Pi_sum <= 0:
                    # Increase precision to avoid all-zero underflow
                    beta = beta * 2
                    continue
                Pi = Pi / Pi_sum

                # Shannon entropy of P(j|i)
                Hi = -torch.sum(Pi * torch.log(Pi + 1e-12))
                Hdiff = Hi - log_perp

                if torch.abs(Hdiff) < tol:
                    break

                if Hdiff > 0:
                    # Entropy too high -> distribution too flat -> increase beta
                    betamin = beta if betamin is None else betamin
                    beta = beta * 2 if betamax is None else 0.5 * (beta + betamax)
                else:
                    # Entropy too low -> distribution too peaky -> decrease beta
                    betamax = beta if betamax is None else betamax
                    beta = beta / 2 if betamin is None else 0.5 * (beta + betamin)

        # Convert beta to sigma: beta = 1 / (2*sigma^2)  => sigma = sqrt(1/(2*beta))
        sigmas[i] = torch.sqrt(1.0 / (2.0 * beta))

        # Final weights on the KNN edges for node i
        if use_probabilities:
            Pi = torch.exp(-Di * beta)
            Pi = Pi / (Pi.sum() + 1e-12)   # row-stochastic on KNN
            wi = Pi
        else:
            wi = torch.exp(-Di * beta)     # raw Gaussian kernel (not normalized)

        # Accumulate COO entries
        rows.append(torch.full((k,), i, device=device, dtype=torch.long))
        cols.append(idx.to(torch.long))
        vals.append(wi.to(dtype))

    rows = torch.cat(rows)
    cols = torch.cat(cols)
    vals = torch.cat(vals)

    W = torch.sparse_coo_tensor(
        indices=torch.stack([rows, cols], dim=0),
        values=vals,
        size=(N, N),
        device=device,
        dtype=dtype,
    ).coalesce()

    # Optional symmetrization
    if sym_mode in ("mean", "sum"):
        WT = torch.sparse_coo_tensor(
            indices=torch.stack([cols, rows], dim=0),
            values=vals,
            size=(N, N),
            device=device,
            dtype=dtype,
        ).coalesce()
        if sym_mode == "mean":
            W = (W + WT) * 0.5
        else:  # "sum"
            W = (W + WT)
        W = W.coalesce()

    return W, sigmas, nn_indices

# Functions to compute KL divergence and MMD (Maximum Mean Discrepancy)

def kl_divergence(P, Q):
    # P,Q: (n,n) row stochastic transition matrices
    # D_KL(P || Q) = sum_i sum_j P_ij * log(P_ij / Q_ij)
    # Add eps for numerical stability
    eps = 1e-8

    # Convert to dense if tensor is sparse (output of knn_graph_with_perplexity)
    P = P.to_dense()
    Q = Q.to_dense()
    P = torch.clamp(P, eps, 1)
    Q = torch.clamp(Q, eps, 1)
    return (P * (P.log() - Q.log())).sum(dim=1).mean()

def compute_mmd(x, y, sigma=1.0):
    xx = torch.cdist(x, x, p=2)
    yy = torch.cdist(y, y, p=2)
    xy = torch.cdist(x, y, p=2)
    Kxx = torch.exp(-xx**2/(2*sigma**2)).mean()
    Kyy = torch.exp(-yy**2/(2*sigma**2)).mean()
    Kxy = torch.exp(-xy**2/(2*sigma**2)).mean()
    return Kxx + Kyy - 2*Kxy


# Function for Embedding architecture
class EmbedNet(nn.Module):
    def __init__(self, input_dim, embed_dim=2):
        super().__init__()  
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embed_dim)
        )
    def forward(self, x):
        return self.net(x)
    
# KNN-aware Cauchy kernel
def cauchy_kernel(x, gamma=1.0,neighborindex = None):
    # x: (n,d)
    dist = pairwise_squared_distances(x)
    K = 1 / (1 + dist / gamma)
    K[neighborindex == 0] = 0 # set non-neighbors to 0
    # row normalize to probabilities 
    K = K / K.sum(dim=1, keepdim=True)  
    return K

### Training Loop 
Hey Kiarash, the following code provides the trainingsloop for the Manifold alignment. Please note that I added several comments (highlited by #++++#) on portions where there is still room for experimentation (e.g. which Kernel approach to use in the embedding space). A grander Question would be how the algorithm deals with outliers (here you could say that Achim is cooking smh in this direction currently) and that the optimization paradigms can deal with certain distortions (e.g. MLP can handle linear distortions quite well while pointwise optimization seems generally to be more adaptive in this regard). 

In [ ]:
# Training Loop 

def train_joint_embedding(X, Y, epochs=1000, lr=1e-3, k=20, embed_dim=2, device='cpu', verbose=False, alpha=0.5):

    # Convert to pytorch-tensors datastructure for input matrix R^(n x f): 
    X = torch.tensor(X, dtype=torch.float32, device=device)
    Y = torch.tensor(Y, dtype=torch.float32, device=device)

    # Construction of MLP 
    f_net = EmbedNet(X.shape[1], embed_dim).to(device)
    g_net = EmbedNet(Y.shape[1], embed_dim).to(device)
    optimizer = torch.optim.Adam(list(f_net.parameters()) + list(g_net.parameters()), lr=lr)

    # Adapt K for non-matching spot dimensions to define similar KNN radius: 
    k_P = round(np.sqrt(X.numel() / Y.numel()) * k)
    k_Q = round(np.sqrt(Y.numel() / X.numel()) * k)

    #++++# Here, the choice of kernel is still open for experimentation.
    # Precompute diffusion transition matrices P and Q on X and Y
    P, _ ,_ = knn_graph_with_perplexity(X, k=k_P)
    Q, _ ,_ = knn_graph_with_perplexity(Y, k=k_Q)

    # Training iteration: 
    for epoch in range(epochs):
        optimizer.zero_grad()
        # Put Data matrices through network: 
        fX = f_net(X)
        gY = g_net(Y)

        #++++# Here, the choice of kernel is still open for experimentation.
        # Diffusion on embeddings
        T_fx = cauchy_kernel(fX)
        T_gy = cauchy_kernel(gY)

        # Loss functions: 
        kl_loss = kl_divergence(P, T_fx) + kl_divergence(Q, T_gy)
        mmd_loss = compute_mmd(fX, gY, sigma=1.0)

        # Scale losses
        loss = alpha * kl_loss + (1-alpha) * mmd_loss

        # Use this value as backpropagation loss function: 
        loss.backward()
        optimizer.step()
        if (epoch+1) % 100 == 0 and verbose == True:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, KL: {kl_loss.item()*alpha :.4f}, MMD: {mmd_loss.item()* (1-alpha):.4f}")
    return f_net, g_net, loss
    

Sometimes, you want to proceed the trainingsprocess (comes up later) so I constructed a function that allows you to extent the training. Obviousply, this function has to be adapted based on how you altered the train_joint_embeddings() function 

In [76]:
def proceed_train_joint_embedding(X, Y, f_net, g_net, epochs=2000, lr=1e-3, k=20, embed_dim=2, device='cpu', verbose=False, alpha=0.5):

    # Convert to pytorch-tensors datastructure for input matrix R^(n x f): 
    X = torch.tensor(X, dtype=torch.float32, device=device)
    Y = torch.tensor(Y, dtype=torch.float32, device=device)

    optimizer = torch.optim.Adam(list(f_net.parameters()) + list(g_net.parameters()), lr=lr)

    # Adapt K for non-matching spot dimensions to define similar KNN radius: 
    k_P = round(np.sqrt(X.numel() / Y.numel()) * k)
    k_Q = round(np.sqrt(Y.numel() / X.numel()) * k)
    
    # Precompute diffusion transition matrices P and Q on X and Y
    P, _ ,_ = knn_graph_with_perplexity(X, k=k_P)
    Q, _ ,_ = knn_graph_with_perplexity(Y, k=k_Q)

    # Training iteration: 
    for epoch in range(epochs):
        optimizer.zero_grad()
        # Put Data matrices through network: 
        fX = f_net(X)
        gY = g_net(Y)
        # Diffusion on embeddings
        T_fx = cauchy_kernel(fX)
        T_gy = cauchy_kernel(gY)

        # Loss functions: 
        kl_loss = kl_divergence(P, T_fx) + kl_divergence(Q, T_gy)
        mmd_loss = compute_mmd(fX, gY, sigma=1.0)

        # Scale losses
        loss = alpha * kl_loss + (1-alpha) * mmd_loss

        # Use this value as backpropagation loss function: 
        loss.backward()
        optimizer.step()
        if (epoch+1) % 100 == 0 and verbose == True:
            print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}, KL: {kl_loss.item()*alpha :.4f}, MMD: {mmd_loss.item()* (1-alpha):.4f}")
    return f_net, g_net, loss

In [ ]:
# I put a plotting function here to visualize the embeddings in 3D 
# and to distinguish the two datasets by color and shape.
def plot_joint_embeddings_3d(fX_plot, gY_plot, labels_X,labels_Y,point_size=5,
                             title="3D Joint Embedding", cmap_X="Viridis", cmap_Y="Plasma"):

    # Scatter3d traces
    trace_fX = go.Scatter3d(
        x=fX_plot[:, 0], y=fX_plot[:, 1], z=fX_plot[:, 2],
        mode='markers',
        name='f(X)',
        marker=dict(
            size=point_size,
            color=labels_X,
            colorscale=cmap_X,
            opacity=0.8,
            symbol='circle',
            colorbar=dict(title='Labels f(X)')
        )
    )

    trace_gY = go.Scatter3d(
        x=gY_plot[:, 0], y=gY_plot[:, 1], z=gY_plot[:, 2],
        mode='markers',
        name='g(Y)',
        marker=dict(
            size=point_size,
            color=labels_Y, 
            colorscale=cmap_Y,
            symbol='cross',
            opacity=0.8,
            colorbar=dict(title='Labels g(Y)')
        )
    )

    # Layout and figure
    fig = go.Figure(data=[trace_fX, trace_gY])

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis_title='Dim 1',
            yaxis_title='Dim 2',
            zaxis_title='Dim 3'
        ),
        legend=dict(x=0.8, y=0.9),
        margin=dict(l=0, r=0, b=0, t=40)
    )

    fig.write_html("joint_3d_plot.html", auto_open=True)



### Example Run on Swiss Roll 
Here I provide a classic training example (gud old' swissrole) how to execute the algorithm. 
- I first only run the trainingsloop once, but usually, the algoirthm becomes trapped in local optima. By running the algorithms iterativly and only storing the best MLP, you get better solutions

In [ ]:
# Testdataset (Swissrole) to check if the code works: 
from sklearn.datasets import make_swiss_roll # Function for swiss roll
from sklearn.preprocessing import StandardScaler
X, labels = make_swiss_roll(n_samples=1000)

scaler_X = StandardScaler()
X_orig = scaler_X.fit_transform(X)

In [72]:
# Selfallignment 
f_net, g_net, loss = train_joint_embedding(X_orig, X_orig, epochs=1000, lr=1e-3, k=20, embed_dim=3, device='cpu', verbose=True, alpha=0.5)

Epoch 100, Loss: 2.1775, KL: 2.1652, MMD: 0.0123
Epoch 200, Loss: 1.5953, KL: 1.5872, MMD: 0.0081
Epoch 300, Loss: 1.3942, KL: 1.3875, MMD: 0.0067
Epoch 400, Loss: 1.3069, KL: 1.3011, MMD: 0.0058
Epoch 500, Loss: 1.2637, KL: 1.2586, MMD: 0.0050
Epoch 600, Loss: 1.2400, KL: 1.2355, MMD: 0.0045
Epoch 700, Loss: 1.2255, KL: 1.2212, MMD: 0.0043
Epoch 800, Loss: 1.2157, KL: 1.2115, MMD: 0.0042
Epoch 900, Loss: 1.2082, KL: 1.2040, MMD: 0.0042
Epoch 1000, Loss: 1.2020, KL: 1.1977, MMD: 0.0043


In [73]:
# Calculate Embeddings for plotting: 
X = torch.tensor(X_orig, dtype=torch.float32, device="cpu")
Y = X 

fX = f_net(X)
gY = g_net(Y)

fX_np = fX.cpu().detach().numpy()
gY_np = gY.cpu().detach().numpy()

plot_joint_embeddings_3d(fX_np, gY_np, labels_X=labels,labels_Y=labels, point_size=5)

In [ ]:
# Trainings loop (This code snippet takes quite some time!!!)

# Selfalignment: 
X = X_orig
Y = X

# Lists for losses and mean distances: 
loss_function_val = []

X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

nsims = 5
best_loss = np.inf
for i in range(nsims):
    print(f"Current training simulation: {i+1}")
    
    f_net, g_net, loss = train_joint_embedding(X, Y, epochs=3000, verbose=False, k=30, embed_dim=3, alpha=0.03)

    # Construct list of losses: 
    loss_function_val.append(loss.item())
    
    if loss.item() < best_loss:
        bf_net, bg_net = f_net, g_net
        best_loss = loss.item()
        
# Finalize training on best embedding: 
bf_net, bg_net, loss = proceed_train_joint_embedding(X, Y, bf_net, bg_net, epochs=1000, lr=1e-3, k=20, embed_dim=3, device='cpu', verbose=True, alpha=0.03)


0
1
2
Epoch 100, Loss: 0.1251, KL: 0.1194, MMD: 0.0057
Epoch 200, Loss: 0.1231, KL: 0.1171, MMD: 0.0060
Epoch 300, Loss: 0.1209, KL: 0.1149, MMD: 0.0060
Epoch 400, Loss: 0.1189, KL: 0.1129, MMD: 0.0060
Epoch 500, Loss: 0.1171, KL: 0.1111, MMD: 0.0060
Epoch 600, Loss: 0.1155, KL: 0.1095, MMD: 0.0059
Epoch 700, Loss: 0.1141, KL: 0.1082, MMD: 0.0059
Epoch 800, Loss: 0.1127, KL: 0.1070, MMD: 0.0057
Epoch 900, Loss: 0.1111, KL: 0.1060, MMD: 0.0052
Epoch 1000, Loss: 0.1102, KL: 0.1053, MMD: 0.0050


In [80]:
# Plot Embedded Datasets: 
X = torch.tensor(X_orig, dtype=torch.float32, device="cpu")
Y = X 

fX = bf_net(X)
gY = bg_net(Y)

fX_np = fX.cpu().detach().numpy()
gY_np = gY.cpu().detach().numpy()

plot_joint_embeddings_3d(fX_np, gY_np, labels_X=labels,labels_Y=labels, point_size=5)